In [2]:
import os
import json
import base64
from pathlib import Path
from urllib.parse import urlparse

import msal
import requests
from dotenv import load_dotenv

# 1. Load .env from current backend folder
env_path = Path.cwd() / ".env"
if not env_path.exists():
    raise FileNotFoundError(
        f".env not found: {env_path}\n"
        "Run this notebook from the backend directory."
    )

load_dotenv(env_path, override=True)

tenant_id = os.getenv("MS_TENANT_ID")
client_id = os.getenv("MS_CLIENT_ID")
client_secret = os.getenv("MS_CLIENT_SECRET")
site_url = os.getenv("SHAREPOINT_SITE_URL", "").rstrip("/")

if not all([tenant_id, client_id, client_secret, site_url]):
    raise ValueError("One or more required .env values are missing.")

# 2. Initialize MSAL app
app = msal.ConfidentialClientApplication(
    client_id=client_id,
    authority=f"https://login.microsoftonline.com/{tenant_id}",
    client_credential=client_secret,
)

# 3. Acquire a fresh Microsoft Graph token
result = app.acquire_token_for_client(
    scopes=["https://graph.microsoft.com/.default"]
)

if "access_token" not in result:
    print("TOKEN ERROR:", result.get("error"))
    print("DETAIL:", result.get("error_description"))
    raise RuntimeError("Could not acquire Graph token.")

token = result["access_token"]

payload = token.split(".")[1]
payload += "=" * (-len(payload) % 4)
claims = json.loads(base64.urlsafe_b64decode(payload))
roles = claims.get("roles", [])

print("========== TOKEN CHECK ==========")
print("Client ID matches:", claims.get("appid", claims.get("azp")) == client_id)
print("Tenant ID matches:", claims.get("tid") == tenant_id)
print("Graph audience:", claims.get("aud"))
print("Application roles:", roles)
print("Sites.Read.All effective:", "Sites.Read.All" in roles)

# 4. Test SharePoint site access
parsed = urlparse(site_url)
endpoint = f"https://graph.microsoft.com/v1.0/sites/{parsed.hostname}:{parsed.path}"

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
}

response = requests.get(endpoint, headers=headers, timeout=30)

print("\n========== SITE ACCESS ==========")
print("Site HTTP status:", response.status_code)

if not response.ok:
    print("Error:", response.text[:1200])
else:
    site = response.json()
    site_id = site["id"]
    print("SUCCESS: Site accessible")
    print("Site name:", site.get("displayName"))

    # 5. Test document libraries
    drives_response = requests.get(
        f"https://graph.microsoft.com/v1.0/sites/{site_id}/drives",
        headers=headers,
        timeout=30,
    )

    print("\n========== DOCUMENT LIBRARIES ==========")
    print("Drives HTTP status:", drives_response.status_code)

    if drives_response.ok:
        for drive in drives_response.json().get("value", []):
            print("Library:", drive.get("name"))
    else:
        print("Error:", drives_response.text[:1200])

========== TOKEN CHECK ==========
Client ID matches: True
Tenant ID matches: True
Graph audience: https://graph.microsoft.com
Application roles: ['Sites.Selected', 'Sites.Read.All']
Sites.Read.All effective: True

========== SITE ACCESS ==========
Site HTTP status: 200
SUCCESS: Site accessible
Site name: IIIOT INFOTECH PRIVATE LIMITED

========== DOCUMENT LIBRARIES ==========
Drives HTTP status: 200
Library: RPA Documents
Library: Documents
